# Lab 06: Machine Learning with MLlib

## Objectives
- Prepare data for machine learning (feature engineering, scaling)
- Build machine learning pipelines using the Pipeline API
- Train classification models (Logistic Regression, Decision Trees, Random Forest)
- Train regression models (Linear Regression, Gradient Boosting)
- Evaluate model performance using appropriate metrics
- Perform hyperparameter tuning with CrossValidator
- Handle categorical features with StringIndexer and OneHotEncoder
- Use VectorAssembler for feature vector creation

## Domain Coverage
- **MLlib — 10%**

## Estimates
- **Time:** ~90-120 minutes
- **Difficulty:** Advanced
- **Prerequisites:** Lab 01 (Spark Fundamentals), Lab 02 (Data Loading & Transformation)

![MLlib Machine Learning Diagram](../assets/diagrams/lab-06-mllib-machine-learning.mmd)

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.feature import IndexToString
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [ ]:
# Create Spark session for ML
spark = SparkSession.builder \
    .appName("MLlibMachineLearning") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Verify Spark session
print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.conf.get('spark.master')}")

## A. Creating Sample Data for Machine Learning

Let's create comprehensive sample data for classification and regression tasks.

In [ ]:
# Create classification data (customer churn prediction)
classification_data = [
    (1, "Alice", 28, 75000, "Engineering", 5, 0, 0),
    (2, "Bob", 35, 85000, "Marketing", 3, 1, 0),
    (3, "Charlie", 42, 95000, "Engineering", 8, 0, 1),
    (4, "Diana", 31, 80000, "Sales", 4, 2, 0),
    (5, "Eve", 29, 72000, "Marketing", 2, 1, 0),
    (6, "Frank", 38, 88000, "Engineering", 6, 0, 1),
    (7, "Grace", 33, 78000, "Sales", 3, 1, 0),
    (8, "Henry", 45, 105000, "Executive", 10, 0, 1),
    (9, "Ivy", 27, 68000, "Marketing", 1, 2, 0),
    (10, "Jack", 40, 92000, "Engineering", 7, 0, 1),
    (11, "Kate", 36, 82000, "Sales", 4, 1, 0),
    (12, "Leo", 32, 77000, "Engineering", 5, 0, 0),
    (13, "Mary", 39, 90000, "Engineering", 9, 0, 1),
    (14, "Nick", 34, 83000, "Marketing", 4, 1, 0),
    (15, "Olivia", 30, 73000, "Sales", 3, 2, 0)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("years_experience", IntegerType(), True),
    StructField("complaints", IntegerType(), True),
    StructField("churn", IntegerType(), True)  # Target variable
])

df_classification = spark.createDataFrame(classification_data, schema)
print("Classification data created:")
df_classification.show()

# Create regression data (salary prediction)
regression_data = [
    (1, 25, 50000, "Engineering", 1, 55000),
    (2, 30, 60000, "Marketing", 3, 65000),
    (3, 35, 70000, "Engineering", 5, 75000),
    (4, 40, 80000, "Sales", 7, 85000),
    (5, 28, 55000, "Marketing", 2, 60000),
    (6, 45, 95000, "Engineering", 15, 105000),
    (7, 32, 68000, "Sales", 4, 73000),
    (8, 38, 82000, "Engineering", 8, 87000),
    (9, 27, 58000, "Marketing", 2, 63000),
    (10, 42, 88000, "Engineering", 10, 93000)
]

regression_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("education_level", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("years_experience", IntegerType(), True),
    StructField("salary", IntegerType(), True)  # Target variable
])

df_regression = spark.createDataFrame(regression_data, regression_schema)
print("\nRegression data created:")
df_regression.show()

## B. Data Preparation for Classification

Prepare data for machine learning by handling categorical features and creating feature vectors.

In [ ]:
# Split data into train and test
train_data, test_data = df_classification.randomSplit([0.8, 0.2], seed=42)
print(f"Training data: {train_data.count()} rows")
print(f"Test data: {test_data.count()} rows")

# Index categorical features
department_indexer = StringIndexer(inputCol="department", outputCol="department_index")

# Create feature vector
assembler = VectorAssembler(
    inputCols=["age", "salary", "years_experience", "complaints", "department_index"],
    outputCol="features"
)

# Scale features
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

print("Data preparation components created")

## C. Building Classification Pipelines

Create machine learning pipelines for classification tasks.

In [ ]:
# Logistic Regression Pipeline
lr = LogisticRegression(featuresCol="scaled_features", labelCol="churn")

pipeline_lr = Pipeline(stages=[department_indexer, assembler, scaler, lr])

# Fit the pipeline
model_lr = pipeline_lr.fit(train_data)

# Make predictions
predictions_lr = model_lr.transform(test_data)

print("Logistic Regression predictions:")
predictions_lr.select("churn", "prediction", "probability").show(5, truncate=False)

In [ ]:
# Decision Tree Pipeline
dt = DecisionTreeClassifier(featuresCol="scaled_features", labelCol="churn")

pipeline_dt = Pipeline(stages=[department_indexer, assembler, scaler, dt])

# Fit the pipeline
model_dt = pipeline_dt.fit(train_data)

# Make predictions
predictions_dt = model_dt.transform(test_data)

print("Decision Tree predictions:")
predictions_dt.select("churn", "prediction").show(5)

In [ ]:
# Random Forest Pipeline
rf = RandomForestClassifier(featuresCol="scaled_features", labelCol="churn", numTrees=10)

pipeline_rf = Pipeline(stages=[department_indexer, assembler, scaler, rf])

# Fit the pipeline
model_rf = pipeline_rf.fit(train_data)

# Make predictions
predictions_rf = model_rf.transform(test_data)

print("Random Forest predictions:")
predictions_rf.select("churn", "prediction").show(5)

## D. Model Evaluation for Classification

Evaluate classification models using appropriate metrics.

In [ ]:
# Evaluate models
evaluator = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC")

auc_lr = evaluator.evaluate(predictions_lr)
auc_dt = evaluator.evaluate(predictions_dt)
auc_rf = evaluator.evaluate(predictions_rf)

print(f"Logistic Regression AUC: {auc_lr:.4f}")
print(f"Decision Tree AUC: {auc_dt:.4f}")
print(f"Random Forest AUC: {auc_rf:.4f}")

# Additional metrics
evaluator_accuracy = BinaryClassificationEvaluator(labelCol="churn", metricName="accuracy")

accuracy_lr = evaluator_accuracy.evaluate(predictions_lr)
print(f"Logistic Regression Accuracy: {accuracy_lr:.4f}")

## E. Regression Pipeline

Build a regression pipeline for predicting continuous values.

In [ ]:
# Split regression data
train_reg, test_reg = df_regression.randomSplit([0.8, 0.2], seed=42)

# Index categorical feature
dept_indexer_reg = StringIndexer(inputCol="department", outputCol="department_index")

# Create feature vector
assembler_reg = VectorAssembler(
    inputCols=["age", "education_level", "years_experience", "department_index"],
    outputCol="features"
)

# Linear Regression
lr_reg = LinearRegression(featuresCol="features", labelCol="salary")

# Pipeline
pipeline_reg = Pipeline(stages=[dept_indexer_reg, assembler_reg, lr_reg])

# Fit pipeline
model_reg = pipeline_reg.fit(train_reg)

# Make predictions
predictions_reg = model_reg.transform(test_reg)

print("Regression predictions:")
predictions_reg.select("salary", "prediction").show(5)

## F. Regression Model Evaluation

Evaluate regression models using appropriate metrics.

In [ ]:
# Evaluate regression model
evaluator_reg = RegressionEvaluator(labelCol="salary", predictionCol="prediction")

rmse = evaluator_reg.setMetricName("rmse").evaluate(predictions_reg)
mae = evaluator_reg.setMetricName("mae").evaluate(predictions_reg)
r2 = evaluator_reg.setMetricName("r2").evaluate(predictions_reg)

print(f"Root Mean Square Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2): {r2:.4f}")

## G. Hyperparameter Tuning

Use CrossValidator to find optimal hyperparameters.

In [ ]:
# Create parameter grid for Logistic Regression
param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .addGrid(lr.maxIter, [10, 50, 100]) \
    .build()

# Create cross validator
crossval = CrossValidator(
    estimator=pipeline_lr,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3
)

# Run cross validation
cv_model = crossval.fit(train_data)

# Best model
best_model = cv_model.bestModel
print("Best model parameters:")
print(best_model.stages[-1].extractParamMap())

# Evaluate best model
best_predictions = best_model.transform(test_data)
best_auc = evaluator.evaluate(best_predictions)
print(f"Best model AUC: {best_auc:.4f}")

## H. Model Persistence

Save and load trained models for future use.

In [ ]:
# Save model
model_lr.write().overwrite().save("models/logistic_regression")
print("Model saved to models/logistic_regression")

# Load model
from pyspark.ml.pipeline import PipelineModel
loaded_model = PipelineModel.load("models/logistic_regression")

# Use loaded model
loaded_predictions = loaded_model.transform(test_data)
print("Model loaded and predictions generated")
loaded_predictions.select("churn", "prediction").show(5)

## Exam-Style Review

**Q1.** What is the purpose of VectorAssembler in MLlib?
- A) To scale features
- B) To combine multiple feature columns into a single feature vector
- C) To handle categorical features
- D) To split data into train and test

**Q2.** What is the difference between Transformer and Estimator in MLlib?
- A) Transformer fits data, Estimator transforms data
- B) Transformer transforms data, Estimator fits on data
- C) They are identical
- D) Estimator is for classification only

**Q3.** Which metric would you use for evaluating a regression model?
- A) Accuracy
- B) AUC-ROC
- C) RMSE (Root Mean Square Error)
- D) F1 Score

**Q4.** What is the purpose of CrossValidator?
- A) To create cross-validation folds for hyperparameter tuning
- B) To validate data quality
- C) To convert data types
- D) To handle missing values

### Answers
- **Q1: B** — VectorAssembler combines multiple feature columns into a single feature vector column required by ML algorithms.
- **Q2: B** — Transformer transforms data (e.g., feature scaling), while Estimator fits on data (e.g., model training).
- **Q3: C** — RMSE is used for regression models to measure the average magnitude of prediction errors.
- **Q4: A** — CrossValidator performs k-fold cross-validation for hyperparameter tuning and model selection.

## Key Takeaways
- MLlib provides scalable machine learning algorithms for Spark
- Pipeline API chains multiple stages (transformers and estimators)
- StringIndexer and OneHotEncoder handle categorical features
- VectorAssembler creates feature vectors for ML algorithms
- StandardScaler normalizes features for better model performance
- Classification models: Logistic Regression, Decision Trees, Random Forest
- Regression models: Linear Regression, Gradient Boosting
- CrossValidator performs hyperparameter tuning with cross-validation
- Model persistence allows saving and loading trained models

**Next:** [Lab 07 — Production Deployment](chapter-07-production-deployment.ipynb)

In [ ]:
# Cleanup
spark.catalog.clearCache()
print("Lab 06 complete. Cache cleared.")
print("Models saved in models/ directory for future use.")